In [ ]:
#NEW XGBOOST MODEL WITH NEW TRAIN AND TEST DATASET

In [ ]:
"""
XGBoost Model — Small Reptile Species Distribution (EcoStat Modelling, DATA5925)

Trains an XGBoost classifier on the stratified split produced by the train/test split
step. Reads `train_split.csv` for training and `test_split.csv` for evaluation; the test
file is never used during fitting.

Pipeline: load split files -> build features (one-hot Species + numeric predictors) ->
carve a small validation slice from train for early stopping -> train with
scale_pos_weight for class imbalance -> evaluate on the held-out test set
(log loss, AUC, Brier, F1) with a per-species breakdown and feature importance.

Usage:
    python xgboost_model.py
"""

In [9]:

import warnings

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import (accuracy_score, average_precision_score, brier_score_loss,
                             confusion_matrix, f1_score, log_loss, roc_auc_score)
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore", category=UserWarning)

# ----------------------------------------------------------------------------------
# 1. Configuration
# ----------------------------------------------------------------------------------
TARGET = "pres.abs"          # response: 1 = present, 0 = absent
SPECIES = "Species"          # categorical, one-hot encoded
# Plot-level predictors. long/lat and id/plot are excluded (easting/northing carry space).
NUMERIC_FEATURES = ["disturb", "rainann", "soildepth", "soilfert",
                    "tempann", "topo", "easting", "northing"]
RANDOM_STATE = 42
TRAIN_FILE = "train_split.csv"
TEST_FILE = "test_split.csv"


# ----------------------------------------------------------------------------------
# 3. Feature matrix builder
# ----------------------------------------------------------------------------------
def build_X(df, columns=None):
    """One-hot encode Species + keep numeric predictors. Align to `columns` if given."""
    species_dummies = pd.get_dummies(df[SPECIES], prefix="sp")
    numeric_present = [c for c in NUMERIC_FEATURES if c in df.columns]
    X = pd.concat([df[numeric_present].reset_index(drop=True),
                   species_dummies.reset_index(drop=True)], axis=1)
    if columns is not None:                       # align test -> train columns
        X = X.reindex(columns=columns, fill_value=0)
    return X


def main():
    # ------------------------------------------------------------------------------
    # 2. Load the split datasets
    # ------------------------------------------------------------------------------
    train_df = pd.read_csv(TRAIN_FILE)
    test_df = pd.read_csv(TEST_FILE)
    print(f"train: {len(train_df):>5} rows | presence rate {train_df[TARGET].mean():.4f}")
    print(f"test : {len(test_df):>5} rows | presence rate {test_df[TARGET].mean():.4f}")

    # ------------------------------------------------------------------------------
    # 3. Build features (test aligned to train columns)
    # ------------------------------------------------------------------------------
    X_train_full = build_X(train_df)
    y_train_full = train_df[TARGET].astype(int).reset_index(drop=True)
    feature_cols = list(X_train_full.columns)

    X_test = build_X(test_df, columns=feature_cols)
    y_test = test_df[TARGET].astype(int).reset_index(drop=True)
    species_test = test_df[SPECIES].reset_index(drop=True)
    print(f"\n{len(feature_cols)} features: {feature_cols}")

    # ------------------------------------------------------------------------------
    # 4. Train / validation slice for early stopping (test stays untouched)
    # ------------------------------------------------------------------------------
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train_full, y_train_full, test_size=0.15,
        random_state=RANDOM_STATE, stratify=y_train_full)

    n_pos = int(y_tr.sum())
    scale_pos_weight = (len(y_tr) - n_pos) / max(n_pos, 1)
    print(f"\ntrain slice: {len(X_tr)} rows ({n_pos} positives) | "
          f"val slice: {len(X_val)} rows | scale_pos_weight={scale_pos_weight:.2f}")

    # ------------------------------------------------------------------------------
    # 5. Train the XGBoost model
    # ------------------------------------------------------------------------------
    model = xgb.XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        n_estimators=2000,
        learning_rate=0.03,
        max_depth=4,
        min_child_weight=3,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=2.0,
        scale_pos_weight=scale_pos_weight,
        tree_method="hist",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        early_stopping_rounds=50,
    )
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    print(f"best iteration: {model.best_iteration}  (val logloss {model.best_score:.4f})")

    # ------------------------------------------------------------------------------
    # 6. Evaluate on the held-out test set
    # ------------------------------------------------------------------------------
    p = np.clip(model.predict_proba(X_test)[:, 1], 1e-15, 1 - 1e-15)
    pred = (p >= 0.5).astype(int)

    print("\n" + "=" * 45)
    print("TEST PERFORMANCE")
    print("=" * 45)
    print(f"  Log loss (primary) : {log_loss(y_test, p):.4f}")
    print(f"  ROC-AUC            : {roc_auc_score(y_test, p):.4f}")
    print(f"  PR-AUC (avg prec)  : {average_precision_score(y_test, p):.4f}")
    print(f"  Brier score        : {brier_score_loss(y_test, p):.4f}")
    print(f"  F1 (thr=0.5)       : {f1_score(y_test, pred, zero_division=0):.4f}")
    print(f"  Accuracy           : {accuracy_score(y_test, pred):.4f}")
    print("\n  Confusion matrix [[TN, FP], [FN, TP]]:")
    print(confusion_matrix(y_test, pred))

    # ------------------------------------------------------------------------------
    # 7. Per-species performance
    # ------------------------------------------------------------------------------
    res = pd.DataFrame({"species": species_test, "y": y_test, "p": p})
    rows = []
    for sp, g in res.groupby("species"):
        ll = log_loss(g["y"], np.clip(g["p"], 1e-15, 1 - 1e-15), labels=[0, 1])
        auc = roc_auc_score(g["y"], g["p"]) if g["y"].nunique() == 2 else np.nan
        rows.append({"species": sp, "n": len(g), "positives": int(g["y"].sum()),
                     "log_loss": round(ll, 4), "AUC": round(auc, 3)})
    per_species = pd.DataFrame(rows).sort_values("positives", ascending=False).reset_index(drop=True)
    print("\nPER-SPECIES PERFORMANCE")
    print(per_species.to_string(index=False))

    # ------------------------------------------------------------------------------
    # 8. Feature importance
    # ------------------------------------------------------------------------------
    imp = (pd.DataFrame({"feature": feature_cols,
                         "importance": model.feature_importances_})
           .sort_values("importance", ascending=False)
           .reset_index(drop=True))
    print("\nTOP 15 FEATURE IMPORTANCES")
    print(imp.head(15).to_string(index=False))

    # ------------------------------------------------------------------------------
    # 9. Save the trained model
    # ------------------------------------------------------------------------------
    model.save_model("xgboost_species_model.json")
    print("\nSaved xgboost_species_model.json")


if __name__ == "__main__":
    main()

train:  4102 rows | presence rate 0.0631
test :  1026 rows | presence rate 0.0643

16 features: ['disturb', 'rainann', 'soildepth', 'soilfert', 'tempann', 'topo', 'easting', 'northing', 'sp_Cacophis kreftii', 'sp_Calyptotis scutirostrum', 'sp_Coeranoscincus reticulatus', 'sp_Egernia mcpheei', 'sp_Eulamprus murrayi', 'sp_Ophioscincus truncatus', 'sp_Pseudechis porphyricaus', 'sp_Saltuarius swaini']

train slice: 3486 rows (220 positives) | val slice: 616 rows | scale_pos_weight=14.85
best iteration: 1006  (val logloss 0.2135)

TEST PERFORMANCE
  Log loss (primary) : 0.2657
  ROC-AUC            : 0.8685
  PR-AUC (avg prec)  : 0.3789
  Brier score        : 0.0773
  F1 (thr=0.5)       : 0.4176
  Accuracy           : 0.8967

  Confusion matrix [[TN, FP], [FN, TP]]:
[[882  78]
 [ 28  38]]

PER-SPECIES PERFORMANCE
                   species   n  positives  log_loss   AUC
    Ophioscincus truncatus 128         23    0.8126 0.704
Coeranoscincus reticulatus 128         21    0.4968 0.875
   Caly

In [ ]:
"""
Hyperparameter tuning for the XGBoost species model (EcoStat Modelling, DATA5925).

Consistent with xgboost_model.py: trains on `train_split.csv` and only touches
`test_split.csv` for the final check. Different parameter settings are compared by
STRATIFIED K-fold cross-validation on the training file (folds keep the same presence
rate, matching the stratified split philosophy). The test file is never used to choose
parameters.

  * Selection metric: mean cross-validated LOG LOSS (matches the Kaggle log-score);
    mean ROC-AUC reported alongside.
  * scale_pos_weight is recomputed inside every fold from that fold's training rows.
  * Early stopping runs per fold; the mean best iteration sets n_estimators when the
    winning configuration is refit on the full training file.

Search modes:
  --search random  (default) : sample N random combinations from the grid
  --search grid              : try every combination in the grid

Usage:
    python xgboost_tune.py --search random --n-iter 30
    python xgboost_tune.py --search grid
"""

In [10]:

import argparse
import itertools
import warnings

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import log_loss, roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split

warnings.filterwarnings("ignore", category=UserWarning)

# ----------------------------------------------------------------------------------
# Configuration (shared with xgboost_model.py)
# ----------------------------------------------------------------------------------
TARGET = "pres.abs"
SPECIES = "Species"
NUMERIC_FEATURES = ["disturb", "rainann", "soildepth", "soilfert",
                    "tempann", "topo", "easting", "northing"]
RANDOM_STATE = 42
TRAIN_FILE = "train_split.csv"
TEST_FILE = "test_split.csv"
N_SPLITS = 5
EARLY_STOP = 50
MAX_ROUNDS = 3000

# Parameter grid to search. Add or remove values freely.
PARAM_GRID = {
    "max_depth":        [3, 4, 5, 6],
    "learning_rate":    [0.01, 0.03, 0.05, 0.1],
    "min_child_weight": [1, 3, 5],
    "subsample":        [0.7, 0.8, 1.0],
    "colsample_bytree": [0.7, 0.8, 1.0],
    "reg_lambda":       [1.0, 2.0, 5.0],
    "gamma":            [0.0, 0.5, 1.0],
}


# ----------------------------------------------------------------------------------
# Feature building (identical to xgboost_model.py)
# ----------------------------------------------------------------------------------
def build_X(df, columns=None):
    species_dummies = pd.get_dummies(df[SPECIES], prefix="sp")
    numeric_present = [c for c in NUMERIC_FEATURES if c in df.columns]
    X = pd.concat([df[numeric_present].reset_index(drop=True),
                   species_dummies.reset_index(drop=True)], axis=1)
    if columns is not None:
        X = X.reindex(columns=columns, fill_value=0)
    return X


def make_model(params, n_estimators, scale_pos_weight, early_stopping=True):
    return xgb.XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        n_estimators=n_estimators,
        scale_pos_weight=scale_pos_weight,
        tree_method="hist",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        early_stopping_rounds=EARLY_STOP if early_stopping else None,
        **params,
    )


# ----------------------------------------------------------------------------------
# Cross-validated score for one parameter set
# ----------------------------------------------------------------------------------
def cv_score(params, X, y, n_splits):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    losses, aucs, best_iters = [], [], []

    for tr_idx, val_idx in skf.split(X, y):
        X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]
        X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

        n_pos = int(y_tr.sum())
        spw = (len(y_tr) - n_pos) / max(n_pos, 1)

        model = make_model(params, MAX_ROUNDS, spw, early_stopping=True)
        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

        p = np.clip(model.predict_proba(X_val)[:, 1], 1e-15, 1 - 1e-15)
        losses.append(log_loss(y_val, p, labels=[0, 1]))
        aucs.append(roc_auc_score(y_val, p) if y_val.nunique() == 2 else np.nan)
        best_iters.append(model.best_iteration + 1)

    return {
        "mean_logloss": float(np.mean(losses)),
        "std_logloss": float(np.std(losses)),
        "mean_auc": float(np.nanmean(aucs)),
        "mean_best_iter": int(np.mean(best_iters)),
    }


def generate_candidates(grid, search, n_iter, seed):
    keys = list(grid.keys())
    full = list(itertools.product(*(grid[k] for k in keys)))
    if search == "grid":
        combos = full
    else:
        rng = np.random.default_rng(seed)
        n = min(n_iter, len(full))
        combos = [full[i] for i in rng.choice(len(full), size=n, replace=False)]
    return [dict(zip(keys, c)) for c in combos]


# ----------------------------------------------------------------------------------
# Main
# ----------------------------------------------------------------------------------
def main():
    parser = argparse.ArgumentParser(description="XGBoost hyperparameter tuning.")
    parser.add_argument("--train", default=TRAIN_FILE)
    parser.add_argument("--test", default=TEST_FILE)
    parser.add_argument("--search", choices=["random", "grid"], default="random")
    parser.add_argument("--n-iter", type=int, default=30,
                        help="Number of random configs (ignored for grid search)")
    parser.add_argument("--n-splits", type=int, default=N_SPLITS)
    parser.add_argument("--top", type=int, default=10)
    args, _ = parser.parse_known_args()  # notebook-safe

    train_df = pd.read_csv(args.train)
    test_df = pd.read_csv(args.test)

    X_train = build_X(train_df)
    y_train = train_df[TARGET].astype(int).reset_index(drop=True)
    feature_cols = list(X_train.columns)
    X_test = build_X(test_df, columns=feature_cols)
    y_test = test_df[TARGET].astype(int).reset_index(drop=True)

    print(f"train: {len(train_df)} rows (presence {y_train.mean():.4f}) | "
          f"test: {len(test_df)} rows (presence {y_test.mean():.4f})")

    candidates = generate_candidates(PARAM_GRID, args.search, args.n_iter, RANDOM_STATE)
    print(f"\n{args.search.capitalize()} search over {len(candidates)} configs, "
          f"{args.n_splits}-fold stratified CV on the training file ...\n")

    results = []
    for i, params in enumerate(candidates, 1):
        score = cv_score(params, X_train, y_train, args.n_splits)
        results.append({**params, **score})
        print(f"  [{i:>3}/{len(candidates)}] "
              f"logloss={score['mean_logloss']:.4f} (+/-{score['std_logloss']:.4f})  "
              f"auc={score['mean_auc']:.3f}  "
              f"depth={params['max_depth']} lr={params['learning_rate']} "
              f"mcw={params['min_child_weight']} sub={params['subsample']} "
              f"col={params['colsample_bytree']} lam={params['reg_lambda']} "
              f"gam={params['gamma']}")

    res_df = pd.DataFrame(results).sort_values("mean_logloss").reset_index(drop=True)

    print("\n" + "=" * 72)
    print(f"TOP {args.top} CONFIGURATIONS BY MEAN CV LOG LOSS")
    print("=" * 72)
    cols = ["mean_logloss", "std_logloss", "mean_auc", "mean_best_iter",
            "max_depth", "learning_rate", "min_child_weight",
            "subsample", "colsample_bytree", "reg_lambda", "gamma"]
    with pd.option_context("display.width", 200, "display.max_columns", None):
        print(res_df[cols].head(args.top).to_string(index=False))

    res_df.to_csv("tuning_results.csv", index=False)
    print("\nSaved full results to tuning_results.csv")

    # ---- Refit the best configuration on the full training file, score on test ----
    best = res_df.iloc[0]
    int_keys = {"max_depth", "min_child_weight"}
    best_params = {
        k: (int(best[k]) if k in int_keys else float(best[k]))
        for k in ["max_depth", "learning_rate", "min_child_weight",
                  "subsample", "colsample_bytree", "reg_lambda", "gamma"]
    }
    n_best = int(best["mean_best_iter"])

    n_pos = int(y_train.sum())
    spw = (len(y_train) - n_pos) / max(n_pos, 1)
    final = make_model(best_params, n_best, spw, early_stopping=False)
    final.fit(X_train, y_train, verbose=False)

    p_test = np.clip(final.predict_proba(X_test)[:, 1], 1e-15, 1 - 1e-15)
    print("\n" + "=" * 72)
    print("BEST CONFIG ON THE HELD-OUT TEST FILE")
    print("=" * 72)
    print(f"  params       : {best_params}")
    print(f"  n_estimators : {n_best}")
    print(f"  CV logloss   : {best['mean_logloss']:.4f}")
    print(f"  TEST logloss : {log_loss(y_test, p_test):.4f}")
    print(f"  TEST ROC-AUC : {roc_auc_score(y_test, p_test):.4f}")

    final.save_model("xgboost_species_best.json")
    print("\nSaved best model to xgboost_species_best.json")


if __name__ == "__main__":
    main()

train: 4102 rows (presence 0.0631) | test: 1026 rows (presence 0.0643)

Random search over 30 configs, 5-fold stratified CV on the training file ...

  [  1/30] logloss=0.2086 (+/-0.0112)  auc=0.912  depth=5 lr=0.05 mcw=1 sub=1.0 col=0.8 lam=5.0 gam=0.5
  [  2/30] logloss=0.1945 (+/-0.0088)  auc=0.910  depth=5 lr=0.01 mcw=3 sub=0.7 col=0.8 lam=1.0 gam=1.0
  [  3/30] logloss=0.2068 (+/-0.0108)  auc=0.906  depth=4 lr=0.05 mcw=5 sub=0.8 col=1.0 lam=5.0 gam=0.5
  [  4/30] logloss=0.2169 (+/-0.0104)  auc=0.909  depth=6 lr=0.03 mcw=3 sub=1.0 col=1.0 lam=2.0 gam=1.0
  [  5/30] logloss=0.2233 (+/-0.0061)  auc=0.908  depth=3 lr=0.03 mcw=3 sub=0.7 col=1.0 lam=1.0 gam=1.0
  [  6/30] logloss=0.2190 (+/-0.0116)  auc=0.908  depth=3 lr=0.03 mcw=3 sub=0.8 col=0.8 lam=2.0 gam=0.5
  [  7/30] logloss=0.2167 (+/-0.0110)  auc=0.905  depth=3 lr=0.05 mcw=5 sub=1.0 col=0.7 lam=5.0 gam=0.0
  [  8/30] logloss=0.1844 (+/-0.0108)  auc=0.910  depth=6 lr=0.03 mcw=1 sub=0.8 col=0.8 lam=1.0 gam=1.0
  [  9/30] logloss